In [ ]:
import pandas as pd

In [ ]:
BATCH = "260618"

In [ ]:
gt_df = pd.read_csv(f"../data/external/boll_emp_by_date.csv", encoding="utf-8", index_col=[0,2], dtype={"date": str})
# gt_df["short_title"] = gt_df["short_title_edition"].apply(lambda x: x[:-5]).str.strip()

In [ ]:
gt_df.head()

In [ ]:
raw_output_df = pd.read_csv(f"../data/processed/batch_{BATCH}/{BATCH}_postproc.csv", encoding="utf-8-sig").iloc[2:].drop(columns="unclassified_text").dropna(subset="Shelfmark").set_index(["Shelfmark", "edition"])
raw_output_df

In [ ]:
gt_df

In [ ]:
raw_output_df.index.dropna()

In [ ]:
# raw_output_df.reindex(gt_df.index).dropna(how="all").to_csv("../data/processed/EMP_batch1.csv", encoding="utf-8-sig")
# raw_output_df.reindex(gt_df.index).dropna(how="all").to_excel("../data/processed/EMP_batch1.xlsx")

### Check what's missing from each index

In [ ]:
idx = pd.Index(gt_df["Shelfmark"])
idx.difference(raw_output_df.index), raw_output_df.index.difference(idx)

In [ ]:
idx_clean = idx.map(lambda x: {"Jav.70": "Jav. 70"}.get(x, x))
idx_clean.difference(raw_output_df.index)

In [ ]:
# output_df.loc[("San Guo", "1892"), "Date 1"] = "1892"
# output_df.loc[("San Guo", "1892"), "Method of acquisition"] = "legal deposit"

In [ ]:
raw_output_df.loc[idx_clean].dropna(axis=1, how="all").columns.difference(gt_df.dropna(axis=1, how="all").columns)

In [ ]:
gt_df.dropna(axis=1, how="all").columns.difference(raw_output_df.loc[idx_clean].dropna(axis=1, how="all").columns)

In [ ]:
gt_df["Language note"]

In [ ]:
output_df = raw_output_df.loc[idx_clean].drop(columns="unclassified_text").reset_index()

In [ ]:
# output_df.to_csv("../data/processed/non_gt_outputs/batch_260130/260130_extracted_trial_records.csv", encoding="utf-8-sig", index=False)

1. Check how the columns I outputted compared to AG's columns
2. Refresh on which of AG's other columns I could generate
    - Potentially:
        - Illustrations
        - Language note
        - Manufacturer
        - Place of manufacture
        - Split date1/date2

In [ ]:
(output_df + "||" + gt_df)["Date 1"][~(output_df["Date 1"] == gt_df["Date 1"])]

In [ ]:
matching_fractions = {}
for c in output_df.dropna(axis=1, how="all").columns:
    diff = (output_df[c].astype(str) + "||" + gt_df[c].astype(str))[~(output_df[c] == gt_df[c])]
    matching_fractions[c] = sum(output_df[c] == gt_df[c]) / len(output_df)
    if not diff.empty:
        print(c + "\n")
        print(diff)
        print("\n")

In [ ]:
pd.DataFrame.from_records(matching_fractions, index=[BATCH])